# Raw H5 Data → LeRobotDataset v2.1 Direct Conversion

이 노트북은 raw H5 파일을 직접 LeRobotDataset **v2.1 형식(episode-per-file)**으로 변환합니다.

v3를 거치지 않고 한 번에 변환하므로 더 효율적입니다.

핵심 목표:
- Raw H5 파일 읽기
- Image 데이터를 OpenPI-safe uint8 고정크기 텐서로 변환
- v2.1 폴더 레이아웃(`meta/`, `data/chunk-{}/`) 생성
- 메타데이터(info.json, episodes.jsonl, tasks.jsonl) 작성

## 1. 의존성 및 설정 로드

In [ ]:

# 의존성 확인
import sys
import importlib
import yaml
import os

def _req(name: str):
    try:
        return importlib.import_module(name)
    except Exception as e:
        raise RuntimeError(f'Missing dependency: {name} ({e})')

np = _req('numpy')
h5py = _req('h5py')
pa = _req('pyarrow')
pq = _req('pyarrow.parquet')
PIL = _req('PIL')
Image = _req('PIL.Image')
tqdm = _req('tqdm.auto').tqdm
cv2 = _req('cv2')

print('python:', sys.executable)
print('numpy:', np.__version__)
print('h5py:', h5py.__version__)
print('pyarrow:', pa.__version__)
print('Pillow:', PIL.__version__)
print('opencv:', cv2.__version__)


## 2. 경로 및 설정

In [ ]:

from pathlib import Path
import json
import shutil
import logging

# config.yaml에서 설정 로드
with open(r'../config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

root = config['demo_root']
task_name = config['conversion_task_name']
demo_root = os.path.join(root, task_name)
user_name = config['user_name']
v2_dataset_saving_root = config['v2_dataset_saving_root']
conversion_prompt = config.get('conversion_prompt', 'pick up the cup and place it on the plate')

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 경로 설정
RAW_H5_DIR = Path(demo_root)
OUTPUT_V21_DATASET_DIR = Path(os.path.join(v2_dataset_saving_root, task_name))
os.makedirs(OUTPUT_V21_DATASET_DIR, exist_ok=True)

# v2.1 설정
REPO_ID = f"{user_name}/{task_name}"
DATASET_FPS = config['rec_fps']
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
CHUNKS_SIZE = 1000  # v2.1 에피소드 청크 크기

# ─────────────────────────────────────────────────────────────
# 증분 처리 모드 플래그
#   True  → 이미 변환된 에피소드는 건너뛰고 새 에피소드만 처리
#   False → 기존 데이터셋을 전부 삭제하고 처음부터 전체 재변환
# ─────────────────────────────────────────────────────────────
INCREMENTAL_MODE = config.get('incremental_mode', True)

print('RAW_H5_DIR         :', RAW_H5_DIR)
print('OUTPUT_V21_DATASET_DIR:', OUTPUT_V21_DATASET_DIR)
print('DATASET_FPS        :', DATASET_FPS)
print('conversion_prompt  :', conversion_prompt)
print('INCREMENTAL_MODE   :', INCREMENTAL_MODE)


## 3. Helper 함수

In [ ]:

import io

def _write_json(path: Path, obj):
    """JSON 파일 작성"""
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))

def _write_jsonl(path: Path, rows: list[dict]):
    """JSONL 파일 작성"""
    path = Path(path)
    with path.open('w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False))
            f.write('\n')

def _resize_with_pad_pil(image: Image.Image, height: int, width: int) -> np.ndarray:
    """
    PIL 이미지를 리사이즈하면서 패딩을 추가 (aspect ratio 유지)
    
    Args:
        image: PIL Image
        height: 목표 높이
        width: 목표 너비
    
    Returns:
        (height, width, 3) numpy array (RGB)
    """
    cur_width, cur_height = image.size
    if cur_width == width and cur_height == height:
        return np.asarray(image, dtype=np.uint8)
    
    # aspect ratio 유지하며 리사이즈
    ratio = max(cur_width / width, cur_height / height)
    resized_height = int(cur_height / ratio)
    resized_width = int(cur_width / ratio)
    resized_image = image.resize((resized_width, resized_height), Image.Resampling.LANCZOS)
    
    # 검은색 패딩 추가
    zero_image = Image.new(resized_image.mode, (width, height), 0)
    pad_height = max(0, int((height - resized_height) / 2))
    pad_width = max(0, int((width - resized_width) / 2))
    zero_image.paste(resized_image, (pad_width, pad_height))
    
    return np.asarray(zero_image, dtype=np.uint8)


def _images_to_fixed_chw_uint8(image_arrays: list, height: int, width: int) -> pa.Array:
    """
    이미지 배열을 OpenPI-safe uint8 고정크기 [C,H,W] 중첩 리스트로 변환
    (padding을 포함한 리사이즈)
    
    Args:
        image_arrays: list of (H, W, C) numpy arrays or PIL Images (RGB)
        height: 목표 높이 (padding 포함)
        width: 목표 너비 (padding 포함)
    
    Returns:
        PyArrow FixedSizeList array [3, H, W]
    """
    chw_list = []
    for img_arr in tqdm(image_arrays, desc='decode+resize images', leave=False):
        # numpy array이면 PIL로 변환, PIL Image면 바로 처리
        if isinstance(img_arr, Image.Image):
            img = img_arr.copy()
        elif isinstance(img_arr, np.ndarray):
            img = Image.fromarray(img_arr.astype(np.uint8))
        else:
            raise TypeError(f'Unexpected image type: {type(img_arr)}')
        
        # RGB로 변환
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # 패딩을 포함한 리사이즈: aspect ratio 유지
        arr = _resize_with_pad_pil(img, height, width)
        if arr.ndim != 3 or arr.shape[2] != 3:
            raise ValueError(f'Expected HWC RGB image, got shape {arr.shape}')
        
        # CHW로 변환
        chw = np.transpose(arr, (2, 0, 1))
        chw_list.append(chw)
    
    # 배열 스택
    if not chw_list:
        # 빈 배열 반환
        values = pa.array([], type=pa.uint8())
        lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
        lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
        lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
        return lvl_c
    
    stacked = np.stack(chw_list, axis=0)  # N,C,H,W
    if stacked.shape[1] != 3 or stacked.shape[2] != height or stacked.shape[3] != width:
        raise ValueError(f'Unexpected stacked shape: {stacked.shape}')
    
    # PyArrow 중첩 FixedSizeList 생성
    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c

def _zeros_fixed_chw_uint8(n: int, height: int, width: int) -> pa.Array:
    """
    0으로 채워진 OpenPI-safe uint8 [C,H,W] 배열 생성 (길이 n)
    """
    if n <= 0:
        values = pa.array([], type=pa.uint8())
        lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
        lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
        lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
        return lvl_c
    
    stacked = np.zeros((n, 3, height, width), dtype=np.uint8)
    flat = stacked.reshape(-1).tolist()
    values = pa.array(flat, type=pa.uint8())
    lvl_w = pa.FixedSizeListArray.from_arrays(values, width)
    lvl_h = pa.FixedSizeListArray.from_arrays(lvl_w, height)
    lvl_c = pa.FixedSizeListArray.from_arrays(lvl_h, 3)
    return lvl_c


def _write_video(
    image_arrays: list,
    output_path: Path,
    fps: float,
    height: int,
    width: int,
) -> None:
    """
    이미지 리스트를 MP4 비디오로 저장합니다.

    Args:
        image_arrays: list of (H, W, C) numpy arrays or PIL Images (RGB)
        output_path: 저장할 .mp4 경로
        fps: 비디오 FPS
        height: 출력 프레임 높이
        width: 출력 프레임 너비
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, float(fps), (width, height))

    try:
        for img_arr in image_arrays:
            if isinstance(img_arr, Image.Image):
                img = img_arr
            elif isinstance(img_arr, np.ndarray):
                img = Image.fromarray(img_arr.astype(np.uint8))
            else:
                raise TypeError(f'Unexpected image type: {type(img_arr)}')

            if img.mode != 'RGB':
                img = img.convert('RGB')

            # aspect ratio 유지 리사이즈 + 패딩
            arr_rgb = _resize_with_pad_pil(img, height, width)  # HWC RGB
            arr_bgr = cv2.cvtColor(arr_rgb, cv2.COLOR_RGB2BGR)
            writer.write(arr_bgr)
    finally:
        writer.release()


print('Helper functions loaded.')


## 4. 메인 변환 함수

In [ ]:

def _load_existing_metadata(dataset_dir: Path):
    """
    기존 데이터셋의 메타데이터를 로드합니다.

    Returns:
        episodes_jsonl_rows  : list[dict] — 기존 episodes.jsonl 내용
        existing_episode_idx : set[int]  — 이미 변환된 episode_index 집합
        total_frames_so_far  : int       — 기존 총 프레임 수
    """
    episodes_path = dataset_dir / 'meta' / 'episodes.jsonl'
    episodes_jsonl_rows = []
    existing_episode_idx = set()

    if episodes_path.exists():
        with episodes_path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                episodes_jsonl_rows.append(row)
                existing_episode_idx.add(int(row['episode_index']))

    # 기존 총 프레임 수 = 각 에피소드 length 합산
    total_frames_so_far = sum(r.get('length', 0) for r in episodes_jsonl_rows)
    return episodes_jsonl_rows, existing_episode_idx, total_frames_so_far


def convert_h5_to_v21_dataset(incremental: bool | None = None):
    """
    Raw H5 파일을 직접 LeRobotDataset v2.1 형식으로 변환합니다.

    Args:
        incremental: 증분 처리 여부.
                     None이면 셀 설정의 INCREMENTAL_MODE 값을 사용합니다.
                     True  → 이미 변환된 에피소드는 건너뛰고 새 에피소드만 처리
                     False → 기존 데이터셋 전부 삭제 후 전체 재변환
    """
    _incremental = INCREMENTAL_MODE if incremental is None else incremental
    logging.info(f"변환 시작 (incremental={_incremental})")
    logging.info(f"입력 경로: '{RAW_H5_DIR}'")
    logging.info(f"출력 경로: '{OUTPUT_V21_DATASET_DIR}'")

    # ─── 비증분 모드: 기존 데이터셋 삭제 후 초기화 ─────────────────
    if not _incremental:
        if OUTPUT_V21_DATASET_DIR.exists():
            logging.info(f"[전체 재변환] 기존 디렉토리 삭제 중: '{OUTPUT_V21_DATASET_DIR}'")
            shutil.rmtree(OUTPUT_V21_DATASET_DIR)
        episodes_jsonl_rows = []
        existing_episode_idx = set()
        total_frames = 0
    else:
        # ─── 증분 모드: 기존 메타데이터 로드 ───────────────────────
        if OUTPUT_V21_DATASET_DIR.exists():
            episodes_jsonl_rows, existing_episode_idx, total_frames = _load_existing_metadata(OUTPUT_V21_DATASET_DIR)
            logging.info(
                f"[증분 처리] 기존 에피소드 {len(existing_episode_idx)}개 "
                f"({total_frames} 프레임) 감지 — 건너뜀"
            )
        else:
            episodes_jsonl_rows = []
            existing_episode_idx = set()
            total_frames = 0

    # 디렉토리 생성
    (OUTPUT_V21_DATASET_DIR / 'meta').mkdir(parents=True, exist_ok=True)
    (OUTPUT_V21_DATASET_DIR / 'data').mkdir(parents=True, exist_ok=True)
    (OUTPUT_V21_DATASET_DIR / 'videos').mkdir(parents=True, exist_ok=True)

    # 비디오 스트림 키 정의 (video_key → 출력 폴더명)
    VIDEO_STREAMS = {
        'head':  'observation.images.ego_view',
        'left':  'observation.images.left_wrist',
        'right': 'observation.images.right_wrist',
    }

    # v2.1 video feature 공통 정보
    _video_feature = lambda: {
        'dtype': 'video',
        'shape': [IMAGE_HEIGHT, IMAGE_WIDTH, 3],
        'names': ['height', 'width', 'channels'],
        'info': {
            'video.fps': float(DATASET_FPS),
            'video.codec': 'mp4v',
            'video.pix_fmt': 'yuv420p',
            'video.is_depth_map': False,
            'has_audio': False,
        },
    }

    # v2.1 features 정의
    v21_features = {
        'state': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'actions': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'prompt': {'dtype': 'string', 'shape': [1], 'names': None},

        'head_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },
        'left_wrist_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },
        'right_wrist_image': {
            'dtype': 'uint8',
            'shape': [3, IMAGE_HEIGHT, IMAGE_WIDTH],
            'names': ['channels', 'height', 'width'],
        },

        # 비디오 스트림 features
        'observation.images.ego_view':   _video_feature(),
        'observation.images.left_wrist': _video_feature(),
        'observation.images.right_wrist': _video_feature(),

        # LeRobot 표준 필드
        'timestamp': {'dtype': 'float32', 'shape': [1], 'names': None},
        'frame_index': {'dtype': 'int64', 'shape': [1], 'names': None},
        'episode_index': {'dtype': 'int64', 'shape': [1], 'names': None},
        'index': {'dtype': 'int64', 'shape': [1], 'names': None},
        'task_index': {'dtype': 'int64', 'shape': [1], 'names': None},

        # 추가 필드
        'observation.state': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'action': {'dtype': 'float32', 'shape': (16,), 'names': None},
        'is_first': {'dtype': 'bool', 'shape': [1], 'names': None},
        'is_last': {'dtype': 'bool', 'shape': [1], 'names': None},
        'is_terminal': {'dtype': 'bool', 'shape': [1], 'names': None},
    }

    # info.json 기본값
    v21_info = {
        'codebase_version': 'v2.1',
        'robot_type': 'rby1',
        'total_episodes': 0,
        'total_frames': 0,
        'total_tasks': 1,
        'total_chunks': 0,
        'chunks_size': CHUNKS_SIZE,
        'fps': DATASET_FPS,
        'splits': {'train': None},
        'data_path': 'data/chunk-{episode_chunk:03d}/episode_{episode_index:06d}.parquet',
        'video_path': 'videos/chunk-{episode_chunk:03d}/{video_key}/episode_{episode_index:06d}.mp4',
        'features': v21_features,
    }

    # H5 파일 찾기
    h5_files = sorted(list(RAW_H5_DIR.glob("episode_*/*.h5")))
    if not h5_files:
        logging.error(f"'{RAW_H5_DIR}' 하위 episode_*/ 디렉토리에서 .h5 파일을 찾을 수 없습니다.")
        return

    logging.info(f"총 {len(h5_files)}개의 H5 파일을 발견했습니다.")

    # ─── 이미지 키 감지 ──────────────────────────────────────────────
    image_keys_map = {'head': None, 'left': None, 'right': None}
    with h5py.File(h5_files[0], 'r') as f:
        for key in f.keys():
            key_lower = str(key).lower()
            if 'head' in key_lower:
                image_keys_map['head'] = key
            elif 'left' in key_lower:
                image_keys_map['left'] = key
            elif 'right' in key_lower:
                image_keys_map['right'] = key

    logging.info(
        f"감지된 이미지 키: head={image_keys_map['head']}, "
        f"left={image_keys_map['left']}, right={image_keys_map['right']}"
    )

    # ─── 에피소드별 처리 ─────────────────────────────────────────────
    episodes_stats_jsonl_rows = []
    max_chunk_index = -1
    newly_processed = 0
    skipped = 0

    for episode_idx, h5_path in enumerate(h5_files):
        # ── 증분 모드: 이미 변환된 에피소드 건너뜀 ──────────────────
        if _incremental and episode_idx in existing_episode_idx:
            logging.info(f"[SKIP] 에피소드 {episode_idx} ('{h5_path.name}') — 이미 존재")
            skipped += 1
            # 기존 청크 인덱스 추적 유지
            chunk_index = episode_idx // CHUNKS_SIZE
            max_chunk_index = max(max_chunk_index, int(chunk_index))
            continue

        logging.info(
            f"[변환] 에피소드 {episode_idx + 1}/{len(h5_files)}: '{h5_path.name}' 처리 중..."
        )

        try:
            with h5py.File(h5_path, 'r') as f:
                if image_keys_map['head']:
                    num_frames = f[image_keys_map['head']]['image'].shape[0]
                else:
                    raise ValueError("head 이미지를 찾을 수 없습니다.")

                frame_data_dict = {}
                head_image_data_list = []
                left_image_data_list = []
                right_image_data_list = []

                for frame_idx in range(num_frames):
                    frame_data = {}
                    frame_data['is_first'] = np.array([frame_idx == 0], dtype=bool)
                    frame_data['is_last'] = np.array([frame_idx == num_frames - 1], dtype=bool)
                    frame_data['is_terminal'] = np.array([frame_idx == num_frames - 1], dtype=bool)
                    frame_data['prompt'] = conversion_prompt
                    frame_data['frame_index'] = np.array([frame_idx], dtype=np.int64)
                    frame_data['episode_index'] = np.array([episode_idx], dtype=np.int64)
                    frame_data['index'] = np.array([total_frames + frame_idx], dtype=np.int64)
                    frame_data['timestamp'] = np.array([frame_idx / DATASET_FPS], dtype=np.float32)

                    if image_keys_map['head']:
                        head_image_data_list.append(f[image_keys_map['head']]['image'][frame_idx])
                    if image_keys_map['left']:
                        left_image_data_list.append(f[image_keys_map['left']]['image'][frame_idx])
                    if image_keys_map['right']:
                        right_image_data_list.append(f[image_keys_map['right']]['image'][frame_idx])

                    base_state = f['samples/base_state'][frame_idx].astype(np.float32)
                    gripper_state = f['samples/gripper_state'][frame_idx].astype(np.float32)
                    robot_position = f['samples/robot_position'][frame_idx].astype(np.float32)
                    state = np.concatenate([robot_position[8:22], gripper_state]).astype(np.float32)
                    frame_data['observation.state'] = state
                    frame_data['state'] = state

                    gripper_target = f['samples/gripper_target'][frame_idx].astype(np.float32)
                    robot_target_joints = f['samples/robot_target_joints'][frame_idx].astype(np.float32)
                    action = np.concatenate([robot_target_joints[8:22], gripper_target]).astype(np.float32)
                    frame_data['action'] = action
                    frame_data['actions'] = action

                    frame_data_dict[frame_idx] = frame_data

                # 이미지 → PyArrow 배열
                head_image_col = (
                    _images_to_fixed_chw_uint8(head_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH)
                    if head_image_data_list
                    else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                )
                left_image_col = (
                    _images_to_fixed_chw_uint8(left_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH)
                    if left_image_data_list
                    else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                )
                right_image_col = (
                    _images_to_fixed_chw_uint8(right_image_data_list, IMAGE_HEIGHT, IMAGE_WIDTH)
                    if right_image_data_list
                    else _zeros_fixed_chw_uint8(num_frames, IMAGE_HEIGHT, IMAGE_WIDTH)
                )

                # 스칼라 컬럼
                columns = {}
                for key in ['is_first', 'is_last', 'is_terminal', 'timestamp',
                            'frame_index', 'episode_index', 'index']:
                    columns[key] = pa.array([frame_data_dict[i][key] for i in range(num_frames)])
                for key in ['observation.state', 'action', 'state', 'actions']:
                    columns[key] = pa.array([frame_data_dict[i][key] for i in range(num_frames)])

                columns['prompt'] = pa.array([conversion_prompt] * num_frames, type=pa.string())
                columns['task_index'] = pa.array([0] * num_frames, type=pa.int64())
                columns['head_image'] = head_image_col
                columns['left_wrist_image'] = left_image_col
                columns['right_wrist_image'] = right_image_col

                out_tbl = pa.table(columns).replace_schema_metadata(None)

                # Parquet 저장
                chunk_index = episode_idx // CHUNKS_SIZE
                max_chunk_index = max(max_chunk_index, int(chunk_index))
                out_chunk_dir = OUTPUT_V21_DATASET_DIR / 'data' / f'chunk-{chunk_index:03d}'
                out_chunk_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_chunk_dir / f'episode_{episode_idx:06d}.parquet'
                pq.write_table(out_tbl, out_path, compression='zstd')
                logging.info(f"  → Parquet 저장 완료: {out_path} ({num_frames} frames)")

                # ─── 비디오 저장 ─────────────────────────────────
                video_source_map = {
                    'head':  head_image_data_list,
                    'left':  left_image_data_list,
                    'right': right_image_data_list,
                }
                for cam_key, img_list in video_source_map.items():
                    if not img_list:
                        continue
                    video_key = VIDEO_STREAMS[cam_key]
                    video_out_dir = (
                        OUTPUT_V21_DATASET_DIR
                        / 'videos'
                        / f'chunk-{chunk_index:03d}'
                        / video_key
                    )
                    video_out_path = video_out_dir / f'episode_{episode_idx:06d}.mp4'
                    _write_video(img_list, video_out_path, DATASET_FPS, IMAGE_HEIGHT, IMAGE_WIDTH)
                    logging.info(f"  → 비디오 저장 완료: {video_out_path}")

                episodes_jsonl_rows.append({
                    'episode_index': int(episode_idx),
                    'length': int(num_frames),
                    'tasks': [conversion_prompt],
                })
                episodes_stats_jsonl_rows.append({
                    'episode_index': int(episode_idx),
                    'stats': {},
                })
                total_frames += num_frames
                newly_processed += 1

        except Exception as e:
            logging.error(f"'{h5_path.name}' 처리 중 오류 발생: {e}", exc_info=True)
            continue

    # ─── 메타데이터 파일 갱신 ────────────────────────────────────────
    # episodes.jsonl 전체를 episode_index 기준으로 정렬하여 재작성
    episodes_jsonl_rows_sorted = sorted(episodes_jsonl_rows, key=lambda r: r['episode_index'])

    total_episodes_final = len(episodes_jsonl_rows_sorted)
    v21_info['total_episodes'] = total_episodes_final
    v21_info['total_frames'] = total_frames
    v21_info['total_chunks'] = max_chunk_index + 1 if max_chunk_index >= 0 else 0
    v21_info['splits']['train'] = f'0:{total_episodes_final}'
    _write_json(OUTPUT_V21_DATASET_DIR / 'meta/info.json', v21_info)
    _write_jsonl(OUTPUT_V21_DATASET_DIR / 'meta/episodes.jsonl', episodes_jsonl_rows_sorted)
    _write_jsonl(OUTPUT_V21_DATASET_DIR / 'meta/tasks.jsonl', [
        {'task_index': 0, 'task': conversion_prompt}
    ])

    # episodes_stats.jsonl — 증분 모드이면 기존 내용에 새 항목만 추가
    stats_path = OUTPUT_V21_DATASET_DIR / 'meta' / 'episodes_stats.jsonl'
    if _incremental and stats_path.exists():
        existing_stats = []
        with stats_path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    existing_stats.append(json.loads(line))
        episodes_stats_jsonl_rows = existing_stats + episodes_stats_jsonl_rows

    _write_jsonl(
        stats_path,
        sorted(episodes_stats_jsonl_rows, key=lambda r: r['episode_index'])
    )

    # ─── 결과 요약 ───────────────────────────────────────────────────
    logging.info("=" * 60)
    logging.info(f"LeRobotDataset v2.1 변환 완료: '{OUTPUT_V21_DATASET_DIR}'")
    logging.info(f"  새로 처리된 에피소드  : {newly_processed}")
    logging.info(f"  건너뛴 에피소드 (기존): {skipped}")
    logging.info(f"  전체 에피소드 (최종)  : {total_episodes_final}")
    logging.info(f"  전체 프레임 (최종)    : {total_frames}")
    logging.info("=" * 60)


# 실행
convert_h5_to_v21_dataset()


## 5. 기존 데이터셋에서 비디오만 생성

이미 parquet으로 변환된 데이터셋에 `videos/` 디렉토리가 없는 경우,
parquet에 저장된 이미지 컬럼(`head_image`, `left_wrist_image`, `right_wrist_image`)을 읽어  
MP4 비디오를 새로 생성합니다. 이미 존재하는 파일은 건너뜁니다.


In [ ]:

# parquet 이미지 컬럼 → HWC numpy 배열 배치 변환 (고속)
def _parquet_col_to_hwc_frames(col, n: int, height: int, width: int) -> np.ndarray:
    """
    PyArrow ChunkedArray(FixedSizeList[3, H, W] uint8)를
    (N, H, W, 3) numpy 배열로 변환합니다.
    """
    # ChunkedArray → 단일 FixedSizeListArray
    arr = col.combine_chunks()
    # 중첩 FixedSizeList 3단계 flatten (C → H → W → flat uint8)
    for _ in range(3):
        arr = arr.flatten()
    np_arr = arr.to_numpy(zero_copy_only=False)
    chw = np_arr[:n * 3 * height * width].reshape(n, 3, height, width)
    return chw.transpose(0, 2, 3, 1)  # (N, H, W, 3)


def generate_videos_from_existing_dataset():
    """
    기존 parquet 데이터셋의 이미지 컬럼으로부터 videos/ 디렉토리를 새로 생성합니다.

    - 이미 존재하는 .mp4는 건너뜀 (증분 처리)
    - info.json의 video_path 필드도 갱신
    """
    VIDEO_STREAMS = {
        'head_image':        'observation.images.ego_view',
        'left_wrist_image':  'observation.images.left_wrist',
        'right_wrist_image': 'observation.images.right_wrist',
    }

    parquet_files = sorted(OUTPUT_V21_DATASET_DIR.glob('data/**/episode_*.parquet'))
    if not parquet_files:
        logging.error(f"'{OUTPUT_V21_DATASET_DIR}' 에서 parquet 파일을 찾을 수 없습니다.")
        return

    logging.info(f"총 {len(parquet_files)}개 parquet 파일에서 비디오 생성 시작")
    generated = 0
    skipped = 0

    for parquet_path in tqdm(parquet_files, desc='비디오 생성'):
        chunk_dir_name = parquet_path.parent.name   # e.g. chunk-000
        episode_stem   = parquet_path.stem           # e.g. episode_000000

        tbl = pq.read_table(parquet_path)
        n   = len(tbl)

        for col_name, video_key in VIDEO_STREAMS.items():
            if col_name not in tbl.column_names:
                continue

            video_out_path = (
                OUTPUT_V21_DATASET_DIR
                / 'videos'
                / chunk_dir_name
                / video_key
                / f'{episode_stem}.mp4'
            )

            if video_out_path.exists():
                skipped += 1
                continue

            video_out_path.parent.mkdir(parents=True, exist_ok=True)

            # parquet CHW 컬럼 → (N, H, W, 3) numpy
            hwc_frames = _parquet_col_to_hwc_frames(
                tbl[col_name], n, IMAGE_HEIGHT, IMAGE_WIDTH
            )

            # MP4 저장 (이미 리사이즈된 프레임이므로 resize 없이 직접 기록)
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(
                str(video_out_path), fourcc, float(DATASET_FPS), (IMAGE_WIDTH, IMAGE_HEIGHT)
            )
            try:
                for frame_hwc in hwc_frames:
                    writer.write(cv2.cvtColor(frame_hwc, cv2.COLOR_RGB2BGR))
            finally:
                writer.release()

            generated += 1

    # info.json video_path 필드 갱신
    info_path = OUTPUT_V21_DATASET_DIR / 'meta' / 'info.json'
    if info_path.exists():
        with info_path.open('r', encoding='utf-8') as f:
            info = json.load(f)
        info['video_path'] = (
            'videos/chunk-{episode_chunk:03d}/{video_key}/episode_{episode_index:06d}.mp4'
        )
        _write_json(info_path, info)
        logging.info("info.json video_path 갱신 완료")

    logging.info(f"비디오 생성 완료: 새로 생성={generated}, 건너뜀={skipped}")


generate_videos_from_existing_dataset()


## 6. 검증

In [ ]:
# 📁 최종 폴더 구조 확인
import os
import glob

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    """재귀적으로 디렉토리 구조 출력"""
    if current_depth >= max_depth:
        return
    
    try:
        entries = sorted(os.listdir(directory))
    except PermissionError:
        return
    
    dirs = [e for e in entries if os.path.isdir(os.path.join(directory, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(directory, e))]
    
    # 파일 출력
    for i, file in enumerate(files):
        is_last = (i == len(files) - 1) and len(dirs) == 0
        symbol = "└── " if is_last else "├── "
        print(f"{prefix}{symbol}{file}")
    
    # 디렉토리 출력
    for i, dir_name in enumerate(dirs):
        is_last = i == len(dirs) - 1
        symbol = "└── " if is_last else "├── "
        print(f"{prefix}{symbol}{dir_name}/")
        
        extension = "    " if is_last else "│   "
        print_tree(os.path.join(directory, dir_name), prefix + extension, max_depth, current_depth + 1)

print("\n" + "="*60)
print("📁 실제 생성된 데이터셋 폴더 구조")
print("="*60)
print(f"\n{OUTPUT_V21_DATASET_DIR}/")
print_tree(str(OUTPUT_V21_DATASET_DIR), max_depth=3)

print("\n" + "="*60)
print("📊 구조 요약")
print("="*60)

meta_dir = OUTPUT_V21_DATASET_DIR / 'meta'
data_dir = OUTPUT_V21_DATASET_DIR / 'data'
video_dir = OUTPUT_V21_DATASET_DIR / 'videos'

# Meta 파일들
meta_files = list(meta_dir.glob('*')) if meta_dir.exists() else []
print(f"\n✅ Meta 디렉토리: {meta_dir.exists()}")
for f in sorted(meta_files):
    print(f"   └─ {f.name}")

# Data 파일들
data_chunks = list(data_dir.glob('chunk-*')) if data_dir.exists() else []
print(f"\n✅ Data 디렉토리: {data_dir.exists()}")
print(f"   총 청크: {len(data_chunks)}")
for chunk in sorted(data_chunks)[:3]:  # 처음 3개만
    parquets = list(chunk.glob('*.parquet'))
    print(f"   └─ {chunk.name}/ ({len(parquets)} parquet files)")

# Videos 폴더
print(f"\n❓ Videos 디렉토리: {video_dir.exists()}")
if not video_dir.exists():
    print(f"   → WRITE_VIDEOS = False이므로 생성되지 않음")
    print(f"   → 이미지는 parquet의 head_image/left_wrist_image/right_wrist_image 컬럼에 포함됨")

print("\n" + "="*60)
print("✅ 최종 데이터셋 형식")
print("="*60)
print(f"""
{task_name}/
├── data/
│   └── chunk-000/
│       ├── episode_000000.parquet
│       ├── episode_000001.parquet
│       └── ... (이미지 데이터 포함: head_image, left_wrist_image, right_wrist_image)
│
├── meta/
│   ├── info.json
│   ├── episodes.jsonl
│   ├── episodes_stats.jsonl
│   └── tasks.jsonl
│
└── (videos/ 폴더는 없음 - 이미지는 parquet에 저장)

📌 주요 특징:
  • 이미지: parquet 컬럼에 OpenPI-safe uint8 nested list [C,H,W] 형식으로 저장
  • 크기: 224x224 (padding 포함, aspect ratio 유지)
  • 컬럼: head_image, left_wrist_image, right_wrist_image
  • 형식: PyArrow FixedSizeList (3차원 배열)
""")


## 7. Hugging Face Hub 업로드

## 7-0. (선택) Hub 커밋 히스토리 초기화

이미 올라간 커밋이 너무 많을 때 히스토리를 정리합니다.

**옵션 A — 레포 삭제 후 단일 커밋으로 전체 재업로드** (빠르고 간단, 히스토리 완전 초기화)  
**옵션 B — git clone → orphan branch squash → force push** (git 히스토리를 커밋 1개로 압축)

아래 셀에서 `RESET_MODE`를 선택하세요.


In [ ]:

import subprocess
import tempfile
import shutil as _shutil
from pathlib import Path as _Path
from huggingface_hub import HfApi, create_repo, upload_folder

# ─────────────────────────────────────────────────────────────────────
# RESET_MODE 선택
#   'delete_reupload' → 레포 삭제 후 로컬 데이터 단일 커밋으로 재업로드 (A 옵션)
#   'squash'          → git clone → orphan branch → force push (B 옵션)
# ─────────────────────────────────────────────────────────────────────
RESET_MODE = 'delete_reupload'   # 'delete_reupload' | 'squash'

_REPO_ID = f"{user_name}/{task_name}"
_api = HfApi()

# ── 옵션 A: 레포 삭제 후 전체 재업로드 ──────────────────────────────
if RESET_MODE == 'delete_reupload':
    print(f'⚠️  레포 삭제 중: {_REPO_ID}')
    _api.delete_repo(repo_id=_REPO_ID, repo_type='dataset')
    print('🗑️  삭제 완료')

    create_repo(repo_id=_REPO_ID, repo_type='dataset', private=False, exist_ok=True)
    print(f'✅ 레포 재생성: https://huggingface.co/datasets/{_REPO_ID}')

    print(f'\n🚀 전체 재업로드 중 (단일 커밋)...')
    res = upload_folder(
        repo_id=_REPO_ID,
        repo_type='dataset',
        folder_path=str(OUTPUT_V21_DATASET_DIR),
        path_in_repo='',
        commit_message='Initial upload (history reset)',
    )
    print(f'✅ 재업로드 완료: {res}')

# ── 옵션 B: git clone → orphan branch squash → force push ────────────
elif RESET_MODE == 'squash':
    _tmp_dir = _Path(tempfile.mkdtemp(prefix='hf_squash_'))
    _clone_dir = _tmp_dir / 'repo'
    _hf_url = f'https://huggingface.co/datasets/{_REPO_ID}'
    print(f'📥 클론 중: {_hf_url} → {_clone_dir}')

    try:
        subprocess.run(['git', 'clone', '--depth', '1', _hf_url, str(_clone_dir)], check=True)

        # orphan 브랜치를 만들어 기존 히스토리를 덮어씀
        subprocess.run(['git', '-C', str(_clone_dir), 'checkout', '--orphan', 'squashed'], check=True)
        subprocess.run(['git', '-C', str(_clone_dir), 'add', '-A'], check=True)
        subprocess.run(
            ['git', '-C', str(_clone_dir), 'commit', '-m', 'Squash: reset history to single commit'],
            check=True,
        )
        # main 브랜치 이름 확인 (main or master)
        _branch = subprocess.check_output(
            ['git', '-C', str(_clone_dir), 'remote', 'show', 'origin'],
        ).decode()
        _main = 'main' if 'HEAD branch: main' in _branch else 'master'

        subprocess.run(
            ['git', '-C', str(_clone_dir), 'branch', '-M', _main],
            check=True,
        )
        print(f'🚀 force push 중...')
        subprocess.run(
            ['git', '-C', str(_clone_dir), 'push', '--force', 'origin', _main],
            check=True,
        )
        print(f'✅ squash force push 완료')

    finally:
        _shutil.rmtree(_tmp_dir, ignore_errors=True)
        print(f'🗑️  임시 디렉토리 삭제: {_tmp_dir}')

else:
    print(f'❌ 알 수 없는 RESET_MODE: {RESET_MODE!r}')
    print("   'delete_reupload' 또는 'squash' 중 하나를 선택하세요.")


In [ ]:

# 사전에 huggingface-cli login 필요
import shutil as _shutil
from pathlib import Path as _Path
from huggingface_hub import HfApi, create_repo, upload_folder, CommitOperationAdd

REPO_ID = f"{user_name}/{task_name}"
PRIVATE = False

api = HfApi()

# 1. 레포지토리 생성 (없으면 새로 만듦)
create_repo(repo_id=REPO_ID, repo_type='dataset', private=PRIVATE, exist_ok=True)
print(f'✅ Repo ready: https://huggingface.co/datasets/{REPO_ID}')

# ─────────────────────────────────────────────────────────────────────
# 2. 업로드 (INCREMENTAL_MODE에 따라 전략 선택)
# ─────────────────────────────────────────────────────────────────────
if not INCREMENTAL_MODE:
    # ── 전체 업로드: 로컬 폴더 전체를 Hub에 덮어씀 (커밋 1개) ──────
    print(f'\n🚀 [전체 업로드] 로컬 → Hub')
    print(f'   로컬: {OUTPUT_V21_DATASET_DIR}')
    print(f'   허브: {REPO_ID}')
    res = upload_folder(
        repo_id=REPO_ID,
        repo_type='dataset',
        folder_path=str(OUTPUT_V21_DATASET_DIR),
        path_in_repo='',
        commit_message='Full re-upload',
    )
    print(f'✅ 전체 업로드 완료: {res}')
else:
    # ── 증분 업로드: Hub에 없는 파일만 커밋 하나로 업로드 ───────────
    print(f'\n🔍 [증분 업로드] Hub에 존재하는 파일 목록 조회 중...')

    try:
        hub_files = {
            item.path
            for item in api.list_repo_tree(
                repo_id=REPO_ID,
                repo_type='dataset',
                recursive=True,
            )
            if hasattr(item, 'path') and not item.path.endswith('/')
        }
        print(f'   Hub 파일 수: {len(hub_files)}')
    except Exception as e:
        print(f'⚠️  Hub 파일 목록 조회 실패 ({e}). 전체 업로드로 폴백합니다.')
        hub_files = set()

    local_root = OUTPUT_V21_DATASET_DIR

    # 업로드할 파일 목록 수집 (parquet + video + meta)
    operations: list[CommitOperationAdd] = []

    # 새 parquet 파일
    for local_path in sorted(local_root.glob('data/**/*.parquet')):
        rel_path = local_path.relative_to(local_root).as_posix()
        if rel_path not in hub_files:
            operations.append(CommitOperationAdd(path_in_repo=rel_path, path_or_fileobj=str(local_path)))

    # 새 video 파일
    for local_path in sorted(local_root.glob('videos/**/*.mp4')):
        rel_path = local_path.relative_to(local_root).as_posix()
        if rel_path not in hub_files:
            operations.append(CommitOperationAdd(path_in_repo=rel_path, path_or_fileobj=str(local_path)))

    # 메타데이터 파일 (항상 최신 버전으로 갱신)
    for meta_file in sorted((local_root / 'meta').iterdir()):
        rel_path = meta_file.relative_to(local_root).as_posix()
        operations.append(CommitOperationAdd(path_in_repo=rel_path, path_or_fileobj=str(meta_file)))

    n_parquets = sum(1 for op in operations if op.path_in_repo.startswith('data/'))
    n_videos   = sum(1 for op in operations if op.path_in_repo.startswith('videos/'))
    n_meta     = sum(1 for op in operations if op.path_in_repo.startswith('meta/'))
    print(f'   새로 업로드할 parquet : {n_parquets}개')
    print(f'   새로 업로드할 video   : {n_videos}개')
    print(f'   메타데이터 갱신       : {n_meta}개')
    print(f'   총 파일               : {len(operations)}개 → 커밋 1개로 처리')

    if operations:
        api.create_commit(
            repo_id=REPO_ID,
            repo_type='dataset',
            operations=operations,
            commit_message=f'Incremental upload: +{n_parquets} parquets, +{n_videos} videos, update meta',
        )
        print(f'✅ 증분 업로드 완료 (단일 커밋)')
    else:
        print('ℹ️  업로드할 새 파일이 없습니다.')

# 3. v2.1 태그 갱신
try:
    api.delete_tag(REPO_ID, tag='v2.1', repo_type='dataset')
    print('🗑️  기존 v2.1 태그 삭제')
except Exception:
    pass

api.create_tag(REPO_ID, tag='v2.1', repo_type='dataset')
print('✅ 태그 v2.1 생성 완료')

# 4. 로컬 HuggingFace 캐시 삭제
hf_cache_path = _Path.home() / '.cache' / 'huggingface' / 'lerobot' / user_name / task_name
if hf_cache_path.exists():
    _shutil.rmtree(hf_cache_path)
    print(f'🗑️  로컬 HF 캐시 삭제: {hf_cache_path}')
else:
    print(f'ℹ️  로컬 HF 캐시 없음 (정상): {hf_cache_path}')

print(f'\n🎉 완료! 학습 재시작 가능')
print(f'   cd /home/hyunjin/rby1_ws/openpi')
print(f'   uv run python scripts/compute_norm_stats.py --config-name pi05_rby1')
print(f'   uv run python scripts/train.py pi05_rby1 --exp-name {task_name}')
